# Training und Kreuzvalidierung der Modelle

## Zweck

Dieses Notebook trainiert und kreuzvalidiert alle Modellkonfigurationen
systematisch als vollständiges Raster (Grid) über die definierten
Hyperparameter. Ergebnisse werden als HTML-Berichte gespeichert.

Bereits vorhandene Berichte werden standardmäßig nicht überschrieben,
sodass das Notebook bei Unterbrechung ohne Datenverlust neu gestartet
werden kann.

## Voraussetzung

Das Notebook zur Aufbereitung der Campus File-Daten muss zuvor ausgeführt
worden sein (`.feather`-Dateien müssen vorliegen).

## Eingabe

| Datei | Beschreibung |
|---|---|
| `grouped_campus_file_pdx.feather` | Aufbereitete Daten (nur Primärdiagnose) |
| `grouped_campus_file_pdx_sdx.feather` | Aufbereitete Daten (inkl. Nebendiagnosen) |
| `columns_metadata.json` | Spaltenbeschreibungen und Datentypen |

## Hyperparameter-Raster

| Parameter | Optionen |
|---|---|
| `icd_structure` | Gruppen- oder Kapitel-Ebene |
| `input_var` | ICD-Codes, Charlson-Score, Elixhauser-Score |
| `use_sdx` | Mit / ohne Nebendiagnosen |
| `kovarianten` | Mit / ohne Kovariaten |
| `output_var` | Verweildauertyp (`typ_vwd`), Entlassungsgrund (`entl_grd`) |
| `subset` | Gesamtdatensatz oder Teilmengen nach Aufnahmeanlass / Partition |

## Ausgabe

- Je eine HTML-Berichtsdatei pro Modellkonfiguration

In [1]:
from drg_risk.pipeline.trainer import Trainer
from drg_risk.data.dataloader import DataLoader
from drg_risk import DATA_DIR, REPORT_DIR

import itertools
import os
from tqdm import tqdm

In [2]:
loader = DataLoader(
    feather_path_pdx =      os.path.join(DATA_DIR, "Campus_File_DRG_2010", "grouped_campus_file_pdx"),
    feather_path_pdx_sdx =  os.path.join(DATA_DIR, "Campus_File_DRG_2010", "grouped_campus_file_pdx_sdx"),     
)

2026-09-24 11:31:10,394 - Dataloader erfolgreich initialisiert.


## Hyperparameterkonfigurationen

Das vollständige kartesische Produkt aller Parameterkombinationen wird
als Iterationsliste aufgebaut. Jede Kombination entspricht einem Modellauf.

In [5]:
icd_structures = ["group", "chapter"]
output_vars = ["typ_vwd", "entl_grd"]
subsets = [None, {"aufn_anl": ["N"]}, {"aufn_anl": ["E", "V", "G"]}, {"partition": ["M"]}, {"partition": ["O"]}]
input_vars = ["charlson", "elixhauser", "codes"]
use_sdx_ = [True, False]
kovarianten_usage = [["typ_alter", "sex", "aufn_anl", "anzahl_nebendiagnosen"], None]

combinations = list(itertools.product(
    use_sdx_, kovarianten_usage, subsets, output_vars, input_vars, icd_structures
))

## Training und Kreuzvalidierung

Für jede Konfiguration wird geprüft, ob der zugehörige Bericht bereits
existiert. Vorhandene Berichte werden übersprungen (`overwrite_existing_reports = False`),
sodass das Notebook bei Unterbrechung wiederaufgenommen werden kann.

In [6]:
for use_sdx, kovarianten, subset, output_var, input_var, icd_struct in tqdm(combinations, desc="cross-validation"):

    # Binärvariable ob existierende Berichte überschrieben werden sollen
    overwrite_existing_reports = False

    # Pfad definieren
    if subset:
        key = list(subset.keys())[0]
        value = list(subset.values())[0]
        value_str = ",".join(str(v) for v in value) if isinstance(value, list) else str(value)
        subset_str = f"subset {key}={value_str}"
    else:
        subset_str = "subset=gesamt"
    path_dir = os.path.join(REPORT_DIR, "scores_postprocessed", f"output_var={output_var}", subset_str, f"input_var={input_var}", f"sdx={use_sdx}")
    kovarianten_bool = bool(kovarianten)
    full_report_path = os.path.join(path_dir, f"{icd_struct}_kovarianten={kovarianten_bool}.html")

    if not overwrite_existing_reports and os.path.exists(full_report_path):
        print(f"Report unter {full_report_path} existiert bereits. Fall wird übersprungen.")
        continue


    trainer = Trainer(
        loader =                        loader,
        output_var =                    output_var,
        icd_structure =                 icd_struct,
        class_weight =                  None,
        load_subset =                   subset,
        input_var =                     input_var,
        kovarianten =                   kovarianten,
        use_sdx =                       use_sdx,
    )

    # Modell kreuzvalidieren
    trainer.cross_validate(n_splits=10)

cross-validation:   0%|          | 0/240 [00:00<?, ?it/s]

cross-validation:  50%|█████     | 121/240 [00:00<00:00, 663.54it/s]

Report unter E:\Uni\Master\007Masterarbeit\drg_risk\Daten\reports\scores_postprocessed\output_var=typ_vwd\subset=gesamt\input_var=charlson\sdx=True\group_kovarianten=True.html existiert bereits. Fall wird übersprungen.
Report unter E:\Uni\Master\007Masterarbeit\drg_risk\Daten\reports\scores_postprocessed\output_var=typ_vwd\subset=gesamt\input_var=charlson\sdx=True\chapter_kovarianten=True.html existiert bereits. Fall wird übersprungen.
Report unter E:\Uni\Master\007Masterarbeit\drg_risk\Daten\reports\scores_postprocessed\output_var=typ_vwd\subset=gesamt\input_var=elixhauser\sdx=True\group_kovarianten=True.html existiert bereits. Fall wird übersprungen.
Report unter E:\Uni\Master\007Masterarbeit\drg_risk\Daten\reports\scores_postprocessed\output_var=typ_vwd\subset=gesamt\input_var=elixhauser\sdx=True\chapter_kovarianten=True.html existiert bereits. Fall wird übersprungen.
Report unter E:\Uni\Master\007Masterarbeit\drg_risk\Daten\reports\scores_postprocessed\output_var=typ_vwd\subset=ges

cross-validation:  77%|███████▋  | 185/240 [00:00<00:00, 388.97it/s]

Report unter E:\Uni\Master\007Masterarbeit\drg_risk\Daten\reports\scores_postprocessed\output_var=typ_vwd\subset aufn_anl=E,V,G\input_var=charlson\sdx=False\group_kovarianten=True.html existiert bereits. Fall wird übersprungen.
Report unter E:\Uni\Master\007Masterarbeit\drg_risk\Daten\reports\scores_postprocessed\output_var=typ_vwd\subset aufn_anl=E,V,G\input_var=charlson\sdx=False\chapter_kovarianten=True.html existiert bereits. Fall wird übersprungen.
Report unter E:\Uni\Master\007Masterarbeit\drg_risk\Daten\reports\scores_postprocessed\output_var=typ_vwd\subset aufn_anl=E,V,G\input_var=elixhauser\sdx=False\group_kovarianten=True.html existiert bereits. Fall wird übersprungen.
Report unter E:\Uni\Master\007Masterarbeit\drg_risk\Daten\reports\scores_postprocessed\output_var=typ_vwd\subset aufn_anl=E,V,G\input_var=elixhauser\sdx=False\chapter_kovarianten=True.html existiert bereits. Fall wird übersprungen.
Report unter E:\Uni\Master\007Masterarbeit\drg_risk\Daten\reports\scores_postpro

cross-validation: 100%|██████████| 240/240 [00:00<00:00, 437.99it/s]

Report unter E:\Uni\Master\007Masterarbeit\drg_risk\Daten\reports\scores_postprocessed\output_var=entl_grd\subset aufn_anl=E,V,G\input_var=charlson\sdx=False\group_kovarianten=False.html existiert bereits. Fall wird übersprungen.
Report unter E:\Uni\Master\007Masterarbeit\drg_risk\Daten\reports\scores_postprocessed\output_var=entl_grd\subset aufn_anl=E,V,G\input_var=charlson\sdx=False\chapter_kovarianten=False.html existiert bereits. Fall wird übersprungen.
Report unter E:\Uni\Master\007Masterarbeit\drg_risk\Daten\reports\scores_postprocessed\output_var=entl_grd\subset aufn_anl=E,V,G\input_var=elixhauser\sdx=False\group_kovarianten=False.html existiert bereits. Fall wird übersprungen.
Report unter E:\Uni\Master\007Masterarbeit\drg_risk\Daten\reports\scores_postprocessed\output_var=entl_grd\subset aufn_anl=E,V,G\input_var=elixhauser\sdx=False\chapter_kovarianten=False.html existiert bereits. Fall wird übersprungen.
Report unter E:\Uni\Master\007Masterarbeit\drg_risk\Daten\reports\scores